In [1]:
import os
import json
import random
import re
import base64
from openai import OpenAI
import anthropic
from dotenv import load_dotenv
import pandas as pd
from tqdm import tqdm
import time


# Load dataset

In [2]:
# Set Paths
base_dir = os.path.join(os.getcwd(), "dataset", "pororo")
qa_json_path = os.path.join(base_dir, "qa.json")
description_csv_path = os.path.join(base_dir, "descriptions.csv")
gif_paths = {}

def load_dataset(qa_json_path, description_csv_path):
    try:
        # Load QA data
        with open(qa_json_path, "r") as f:
            qa_data = json.load(f)["PororoQA"]

        # Load descriptions
        descriptions = pd.read_csv(description_csv_path)

        return qa_data, descriptions
    except Exception as e:
        print(f"Error loading dataset: {e}")
        return [], pd.DataFrame()

def get_random_questions(qa_data, max_questions=40, base_pattern="Pororo_ENGLISH1", seed=42):
    random.seed(seed)

    # Filter all eligible questions
    filtered_questions = [q for q in qa_data if base_pattern in q["video_name"]]

    # Group by unique (video_name, supporting_num) pairs to avoid duplicates
    unique_pairs = {}
    for q in filtered_questions:
        key = (q["video_name"], q["supporting_num"])
        if key not in unique_pairs:
            unique_pairs[key] = []
        unique_pairs[key].append(q)

    # Sample from unique pairs
    unique_keys = list(unique_pairs.keys())
    selected_keys = random.sample(unique_keys, min(max_questions, len(unique_keys)))

    # Get one question from each selected pair
    sampled_questions = []
    episode_counts = {}

    for key in selected_keys:
        question = random.choice(unique_pairs[key])
        sampled_questions.append(question)
        episode = question["video_name"]
        episode_counts[episode] = episode_counts.get(episode, 0) + 1

    # Group by season for display
    season_episodes = {
        "Pororo_ENGLISH1_1": [],
        "Pororo_ENGLISH1_2": [],
        "Pororo_ENGLISH1_3": []
    }

    for ep in episode_counts.keys():
        season = "_".join(ep.split("_")[:3])
        if season in season_episodes:
            season_episodes[season].append(ep)

    # Print statistics
    print(f"\nSelected {len(sampled_questions)} questions from {len(episode_counts)} episodes:")
    for season in sorted(season_episodes.keys()):
        season_eps = {ep: episode_counts[ep] for ep in season_episodes[season]}
        if season_eps:
            print(f"\n{season}:")
            for ep in sorted(season_eps.keys()):
                print(f"  {ep}: {season_eps[ep]} questions")

    return sampled_questions

def get_seeded_question(questions, gif_num, base_seed=42):
    if not questions:
        return None
    # Create new Random instance for each GIF
    local_random = random.Random(base_seed + gif_num)
    # Sort questions to ensure consistent ordering
    sorted_questions = sorted(questions, key=lambda x: x["qid"])
    return local_random.choice(sorted_questions)

def encode_gif(gif_path):
    """Encode GIF file as base64 string"""
    try:
        if not os.path.exists(gif_path):
            print(f"Error: GIF not found at {gif_path}")
            return None
        with open(gif_path, "rb") as gif_file:
            image_base64 = base64.b64encode(gif_file.read()).decode('utf-8')
            return image_base64
    except Exception as e:
        print(f"Error encoding GIF: {e}")
        return None

# Load data
qa_data, descriptions = load_dataset(qa_json_path, description_csv_path)

# Get random sample of questions TODO increase number of questions
sampled_questions = get_random_questions(qa_data, max_questions=40)

# Group questions by supporting_num
grouped_questions = {}
for entry in sampled_questions:
    video_name = entry["video_name"]
    supporting_num = entry["supporting_num"]
    key = (video_name, supporting_num)
    if key not in grouped_questions:
        grouped_questions[key] = []
    grouped_questions[key].append(entry)

# Get unique pairs to process
gif_pairs = sorted(list(grouped_questions.keys()))
correct_count = 0
total_count = len(gif_pairs)

# Used to store question information for each GIF pair
question_data = {}
for video_name, gif_num in gif_pairs:
    current_questions = grouped_questions[(video_name, gif_num)]
    if current_questions:
        entry = get_seeded_question(current_questions, int(gif_num))

        question = entry["question"]
        correct_idx = entry["correct_idx"]
        answers = [entry[f"answer{i}"] for i in range(5)]
        correct_answer = answers[correct_idx]
        qid = entry["qid"]

        question_data[(video_name, gif_num)] = {
            'entry': entry,
            'question': question,
            'correct_answer': correct_answer,
            'qid': qid
        }

        # gif path
        episode_parts = video_name.split("_")
        episode_folder = os.path.join(base_dir, "Scenes_Dialogues",
                                "_".join(episode_parts[:-1]),
                                video_name)
        gif_paths[(video_name, gif_num)] = os.path.join(episode_folder, f"{gif_num}.gif")

results_standard = []




Selected 40 questions from 22 episodes:

Pororo_ENGLISH1_1:
  Pororo_ENGLISH1_1_ep1: 1 questions
  Pororo_ENGLISH1_1_ep10: 2 questions
  Pororo_ENGLISH1_1_ep11: 1 questions
  Pororo_ENGLISH1_1_ep12: 4 questions
  Pororo_ENGLISH1_1_ep13: 2 questions
  Pororo_ENGLISH1_1_ep2: 4 questions
  Pororo_ENGLISH1_1_ep5: 1 questions
  Pororo_ENGLISH1_1_ep6: 4 questions
  Pororo_ENGLISH1_1_ep9: 1 questions

Pororo_ENGLISH1_2:
  Pororo_ENGLISH1_2_ep10: 1 questions
  Pororo_ENGLISH1_2_ep2: 2 questions
  Pororo_ENGLISH1_2_ep5: 2 questions
  Pororo_ENGLISH1_2_ep8: 3 questions

Pororo_ENGLISH1_3:
  Pororo_ENGLISH1_3_ep1: 1 questions
  Pororo_ENGLISH1_3_ep11: 1 questions
  Pororo_ENGLISH1_3_ep12: 2 questions
  Pororo_ENGLISH1_3_ep13: 1 questions
  Pororo_ENGLISH1_3_ep2: 1 questions
  Pororo_ENGLISH1_3_ep3: 1 questions
  Pororo_ENGLISH1_3_ep4: 1 questions
  Pororo_ENGLISH1_3_ep5: 2 questions
  Pororo_ENGLISH1_3_ep7: 2 questions


# Single agent prediction

In [3]:
load_dotenv()

# Configuration
# MODEL_NAME = "gpt-4o-mini"
MODEL_NAME = "claude-3-5-haiku-20241022"

# Determine which platform to use based on the model name
is_openai_model = not MODEL_NAME.startswith("claude-")

# Initialize appropriate client
if is_openai_model:
    client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))
    print(f"Using OpenAI model: {MODEL_NAME}")
else:
    client = anthropic.Anthropic(api_key=os.getenv("ANTHROPIC_API_KEY"))
    print(f"Using Anthropic model: {MODEL_NAME}")

# Load descriptions
descriptions = pd.read_csv(description_csv_path)

def get_prediction(question, image_base64, description, subtitles, max_retries=3, retry_delay=2):
    prompt = f"""
    As a cartoon analysis expert, answer the question using EXACTLY ONE SENTENCE within 30 words.

    Input:
    Question: {question}
    Scene Description: {description}
    Subtitles: {subtitles}

    Guidelines:
    1. Analyze key cartoon elements including character design, facial expressions, compositional framing, color palette, and scene semantics.
    2. No explanations allowed.
    3. Response must be in English only. DO NOT include text in any other language.
    4. NEVER use phrases like "based on...", "according to...", "the description provided...", or "the visual context".
    5. Start your answer directly addressing the question without qualifiers.
    """

    for attempt in range(max_retries):
        try:
            if is_openai_model:
                # OpenAI implementation with image
                completion = client.chat.completions.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url",
                             "image_url": {"url": f"data:image/gif;base64,{image_base64}"}
                            }
                        ]
                    }],
                    max_tokens=50,
                    temperature=0.1,
                )
                response = completion.choices[0].message.content.strip().lower()
            else:
                # Anthropic implementation with image
                completion = client.messages.create(
                    model=MODEL_NAME,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image",
                             "source": {
                                 "type": "base64",
                                 "media_type": "image/gif",
                                 "data": image_base64
                             }
                            }
                        ]
                    }],
                    max_tokens=50,
                    temperature=0.1,
                )
                response = completion.content[0].text.strip().lower()

            # Extract first sentence
            sentences = re.split(r'[.!?]', response)
            first_sentence = sentences[0].strip()

            # Skip empty sentences
            if not first_sentence and len(sentences) > 1:
                first_sentence = next((s.strip() for s in sentences if s.strip()), "")

            return first_sentence

        except Exception as e:
            print(f"Prediction attempt {attempt+1} failed: {e}")
            if attempt < max_retries - 1:
                time.sleep(retry_delay)

    print(f"All {max_retries} attempts failed for question: {question}")
    return None

Using Anthropic model: claude-3-5-haiku-20241022


# Compute accuracy

In [4]:
def compute_accuracy(question, correct_answer, predicted_answer, max_retries=2, retry_delay=2, num_evaluations=3):
    if correct_answer.lower().strip() == predicted_answer.lower().strip():
        return 1.0, [1.0] * num_evaluations
    
    scores = []
    for evaluation_attempt in range(num_evaluations):
        prompt = f"""
        Evaluate the accuracy of the predicted answer according to criteria below:

        Input:
        Question: {question}
        Correct Answer: {correct_answer}
        Predicted Answer: {predicted_answer}

        Evaluation Rules:
        1. Focus PRIMARILY on semantic equivalence.
        2. Additional details should NEVER reduce the score if core information is correct.
        3. Judge based on whether the answer correctly addresses what the question asks for.
        4. Return ONLY a numeric score from [1.0, 0.75, 0.5, 0.25, 0.0].

        Scoring Criteria:
        - 1.0: Contains the correct core information, even if phrased differently or with additional details
        - 0.75: Mostly correct but missing minor information or containing slight inaccuracies
        - 0.5: Partially correct - contains some correct elements but misses important aspects
        - 0.25: Slightly correct - has a small element of the correct answer but is mostly wrong
        - 0.0: Completely incorrect, contradicts the correct answer, or avoids answering

        Scoring Examples:
        - Example of Score 1.0 (Perfect match or semantic equivalence):
        Question: "how did pororo feel after seeing that the flower has wilted"
        Correct: "he was very upset"
        Predicted: "pororo felt sad after seeing that the flower had wilted"
        Score: 1.0 (Synonyms with same core meaning)

        - Example of Score 1.0 (Additional details):
        Question: "what does crong do when pororo says 'come here'"
        Correct: "crong runs away from pororo"
        Predicted: "when pororo says 'come here,' crong tries to run away again"
        Score: 1.0 (Contains core information with additional details)

        - Example of Score 0.75 (Mostly correct but missing or slightly inaccurate information):
        Question: "what did loopy propose to the group after telling them about the flower"
        Correct: "loopy proposed that they should ask her anything"
        Predicted: "loopy proposed to the group that they ask the magic flower questions to predict the future"
        Score: 0.75 (Core action correct but adds slight inaccuracy about asking the flower directly)

        - Example of Score 0.5 (Partially correct):
        Question: "what does pororo almost forget to leave with poby"
        Correct: "the broken camera piece"
        Predicted: "pororo almost forgets to leave with poby's precious camera"
        Score: 0.5 (Mentions camera but misses the specific detail that it's broken)

        - Example of Score 0.25 (Slightly correct):
        Question: "what does eddy ask pororo"
        Correct: "he asks pororo what are you doing"
        Predicted: "eddy asks crong why pororo is acting so urgently"
        Score: 0.25 (Wrong recipient but related to pororo's actions)

        - Example of Score 0.0 (Completely incorrect):
        Question: "what was crong playing with as pororo entered the house"
        Correct: "crong was playing with a snowboard"
        Predicted: "crong was not shown playing with anything"
        Score: 0.0 (Directly contradicts the correct answer)
        """
        for attempt in range(max_retries):
            try:
                if is_openai_model:
                    completion = client.chat.completions.create(
                        model=MODEL_NAME,
                        messages=[{"role": "user", "content": prompt}],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.choices[0].message.content.strip()
                else:
                    completion = client.messages.create(
                        model=MODEL_NAME,
                        messages=[{
                            "role": "user",
                            "content": prompt
                        }],
                        max_tokens=10,
                        temperature=0.1
                    )
                    response = completion.content[0].text.strip()

                numeric_match = re.search(r'(1\.0|0\.75|0\.5|0\.25|0\.0)', response)
                if numeric_match:
                    score = float(numeric_match.group(1))
                else:
                    score = 0.0

                scores.append(score)
                break  

            except Exception as e:
                print(f"Evaluation attempt {evaluation_attempt+1}, retry {attempt+1} failed: {e}")
                if attempt < max_retries - 1:
                    time.sleep(retry_delay)
                    continue
                # If all retries for this evaluation fail, continue to next evaluation

    # If all evaluations failed, return 0
    if not scores:
        return 0.0, []
        
    # Calculate the result using majority voting
    from collections import Counter
    vote_counter = Counter(scores)
    majority_score, count = vote_counter.most_common(1)[0]  # Get most common score
    
    # If there's a tie, calculate average of the tied values
    if len(scores) > 2:  # Only check for ties with more than 2 scores
        top_scores = vote_counter.most_common()
        if len(top_scores) > 1 and top_scores[0][1] == top_scores[1][1]:  # If there's a tie
            # Find all scores with the same count
            tied_scores = [score for score, count in top_scores if count == top_scores[0][1]]
            majority_score = sum(tied_scores) / len(tied_scores)  # Average of tied scores
    
    return majority_score, scores


# Evaluate model performance

In [5]:
try:
    # Initialize counters
    correct_count = 0
    total_count = len(gif_pairs)
    # Process each video and GIF pair
    for video_name, gif_num in tqdm(gif_pairs, total=total_count):
        # Get question information
        if (video_name, gif_num) not in question_data:
            print(f"No question data found for {video_name} GIF {gif_num}")
            continue

        # Use retrieved question information
        q_info = question_data[(video_name, gif_num)]
        question = q_info['question']
        correct_answer = q_info['correct_answer']
        qid = q_info['qid']
        
        # Get gif path
        gif_path = gif_paths[(video_name, gif_num)]
        gif_directory = os.path.dirname(gif_path)
        subtitles_path = os.path.join(gif_directory, "subtitles.txt")

        # Load subtitles
        with open(subtitles_path, "r") as f:
            subtitles = f.read()

        # Get description
        description_rows = descriptions.loc[
            (descriptions.iloc[:, 0] == video_name) &
            (descriptions.iloc[:, 1] == int(gif_num))
        ]
        if description_rows.empty:
            print(f"Description for {video_name} GIF {gif_num} not found")
            continue

        descriptions_list = description_rows.iloc[:, 2].tolist()
        description = " ".join(descriptions_list)

        # Encode GIF image to base64 format
        image_base64 = encode_gif(gif_path)

        if not image_base64:
            print("Error: Could not encode GIF file")
            continue

        # Get prediction
        predicted_answer = get_prediction(question, image_base64, description, subtitles)
        # Calculate accuracy - ensure question parameter is passed
        is_correct = 0
        if predicted_answer is not None:
            is_correct, scores = compute_accuracy(question, correct_answer, predicted_answer)
        correct_count += is_correct
        # Store current result
        result = {
            'gif_num': gif_num,
            'video_name': video_name,
            'qid': qid,
            'question': question,
            'correct_answer': correct_answer,
            'predicted_answer': predicted_answer,
            'scores': scores,  # Store raw scores list
            'evaluator_scores': ','.join([str(score) for score in scores]) if scores else '',  # Pre-format for CSV
            'accuracy': is_correct
        }
        results_standard.append(result)

        print(f"\nVideo name: {video_name}")
        print(f"GIF number: {gif_num}")
        print(f"QID: {qid}")
        print(f"Question: {question}")
        print(f"Correct Answer: {correct_answer}")
        print(f"Predicted Answer: {predicted_answer}")
        if scores:
            print(f"Evaluator Scores: {scores}")
            print(f"Accuracy: {float(is_correct):.4f}")
        else:
            print(f"Warning: No accuracy for question: {question}")
    # Calculate overall accuracy
    average_accuracy = correct_count / total_count if total_count > 0 else 0
    print(f"\nAverage Accuracy: {average_accuracy:.4f}")

except Exception as e:
    print(f"Unexpected error in evaluation: {e}")
    average_accuracy = 0

  2%|▎         | 1/40 [00:19<12:24, 19.08s/it]


Video name: Pororo_ENGLISH1_1_ep1
GIF number: 14
QID: 383
Question: whtat does eddy ask pororo
Correct Answer: he asks pororo what are you doing
Predicted Answer: eddy inquires about pororo's urgent action, seeking an explanation for the penguin's sudden and mysterious behavior
Evaluator Scores: [0.0, 0.0, 0.0]
Accuracy: 0.0000


  5%|▌         | 2/40 [00:38<12:11, 19.24s/it]


Video name: Pororo_ENGLISH1_1_ep10
GIF number: 12
QID: 1100
Question: were eddy's friends interested seeing his new toy?
Correct Answer: yes, they ran happily towards the new toy.
Predicted Answer: pororo and crong eagerly ran toward eddy's new toy car, showing clear excitement and curiosity about the vehicle
Evaluator Scores: [0.0, 0.0, 0.0]
Accuracy: 0.0000


  8%|▊         | 3/40 [01:16<17:17, 28.04s/it]

...


Video name: Pororo_ENGLISH1_3_ep5
GIF number: 25
QID: 2333
Question: does the friends find pororo behind the snow man?
Correct Answer: the friends find pororo behind the snow man.
Predicted Answer: no, the friends do not find pororo behind the snowman in this snowy scene with loopy, poby, crong, and eddy
Evaluator Scores: [0.0, 0.0, 0.0]
Accuracy: 0.0000


 98%|█████████▊| 39/40 [18:03<00:28, 28.45s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 25
QID: 2425
Question: what does crong do when pororo says "come here"
Correct Answer: crong runs away from pororo
Predicted Answer: crong mischievously ignores pororo's command and attempts to run away, displaying his playful and rebellious character typical of young dinosaur characters
Evaluator Scores: [0.0, 0.0, 0.0]
Accuracy: 0.0000


100%|██████████| 40/40 [18:25<00:00, 27.64s/it]


Video name: Pororo_ENGLISH1_3_ep7
GIF number: 47
QID: 2446
Question: what does poby say when invited to play
Correct Answer: he says "of course"
Predicted Answer: poby enthusiastically responds "of course" when invited to play with his friends in the snowy landscape
Evaluator Scores: [0.0, 0.0, 0.0]
Accuracy: 0.0000

Average Accuracy: 0.0000


# Save data

In [6]:
# Remove any existing Average rows
results_standard = [r for r in results_standard if r['gif_num'] != 'Average']

# Get unique videos
unique_videos = len(set(r['video_name'] for r in results_standard))

# Add row numbers to each result
for i, result in enumerate(results_standard, 1):
    result['row_num'] = i

# Add average accuracy as the last row
average_result = {
    'row_num': len(results_standard) + 1,
    'gif_num': 'Average',
    'video_name': f'Total Videos: {unique_videos}',
    'qid': '',
    'question': f'Total Questions: {len(results_standard)}',
    'correct_answer': '',
    'predicted_answer': '',
    'evaluator_scores': '',
    'accuracy': average_accuracy
}
results_standard.append(average_result)

# Define column order (reordered to put video_name before gif_num)
column_order = [
    'row_num',
    'video_name',
    'gif_num',
    'qid',
    'question',
    'correct_answer',
    'predicted_answer',
    'evaluator_scores',
    'accuracy'
]

# Create safe model name for file naming
safe_model_name = MODEL_NAME.replace('-', '_').replace('.', '_')

# Set up output directory
results_dir = os.path.join(os.getcwd(), "results")
os.makedirs(results_dir, exist_ok=True)
# Create standard subdirectory if it doesn't exist
standard_dir = os.path.join(results_dir, "standard")
os.makedirs(standard_dir, exist_ok=True)

output_path = os.path.join(
    results_dir,
    "standard",
    f'pororo_single_agent_{safe_model_name}.csv'
)

# Remove existing file if it exists
if os.path.exists(output_path):
    try:
        os.remove(output_path)
        print(f"Existing file removed: {output_path}")
    except Exception as e:
        print(f"Error removing existing file: {e}")

# Save results with error handling
try:
    results_df = pd.DataFrame(results_standard)
    results_df = results_df[column_order]
    results_df.to_csv(output_path, index=False)

    if os.path.exists(output_path):
        print(f"Results successfully saved to: {output_path}")
    else:
        print(f"Warning: File was not created at {output_path}")
except Exception as e:
    print(f"Error saving results to CSV: {e}")

Existing file removed: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/pororo_single_agent_claude_3_5_haiku_20241022.csv
Results successfully saved to: /Users/wt/PythonProjects/Multi_Agent_Cartoon/results/standard/pororo_single_agent_claude_3_5_haiku_20241022.csv
